In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install xgboost lightgbm catboost --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.6 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from catboost import CatBoostClassifier
import shap

from google.colab import drive
drive.mount('/content/drive')


# PATHS

PROJECT_PATH = "/content/drive/MyDrive/ADHD_EEG_Project"

# INPUT - reuses files already saved by (feature extraction).
# Nothing here is recomputed; this script only READS from this folder.
FEATURES_DIR = os.path.join(PROJECT_PATH, "data", "features_paper_method")

OUTPUT_DIR = os.path.join(PROJECT_PATH, "results_shap_explainability")
os.makedirs(OUTPUT_DIR, exist_ok=True)

MEASURE = "Correlation"   # the best-performing FC measure; change only if
                          # you want to explain a different measure's model

FEATURE_PATH = os.path.join(FEATURES_DIR, f"{MEASURE}_features_paper.npy")
LABELS_PATH = os.path.join(FEATURES_DIR, "labels_paper.npy")
COLUMN_NAMES_PATH = os.path.join(FEATURES_DIR, "column_names_paper.txt")

RFE_N_FEATURES = 20
RFE_STEP = 5
SMOTE_K_NEIGHBORS = 5
RANDOM_STATE = 42


TOP_N_TO_DISPLAY = 10

# LOAD

X = np.load(FEATURE_PATH)
y = np.load(LABELS_PATH)
with open(COLUMN_NAMES_PATH) as f:
    all_column_names = f.read().splitlines()

print(f"Loaded {MEASURE} features: shape {X.shape}")
print(f"Loaded labels: shape {y.shape}, Control={np.sum(y==0)}, ADHD={np.sum(y==1)}")
assert X.shape[1] == len(all_column_names), (
    f"Feature/column-name mismatch: X has {X.shape[1]} columns, "
    f"but column_names_paper.txt has {len(all_column_names)} entries."
)


def make_catboost():
    return CatBoostClassifier(
        iterations=200, depth=6, learning_rate=0.05,
        auto_class_weights="Balanced", random_state=RANDOM_STATE, verbose=0
    )

pipeline = ImbPipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=SMOTE_K_NEIGHBORS)),
    ("scaler", StandardScaler()),
    ("reduce", RFE(
        estimator=make_catboost(),
        n_features_to_select=RFE_N_FEATURES,   # still 20 - unchanged
        step=RFE_STEP
    )),
    ("clf", make_catboost()),
])

print("\nFitting the Correlation + CatBoost (RFE+SMOTE) pipeline on all "
      f"{X.shape[0]} subjects (one-time refit for explainability - see "
      "caveat in the script docstring)...")
pipeline.fit(X, y)
print("done.\n")


X_imputed = pipeline.named_steps["imputer"].transform(X)
X_scaled = pipeline.named_steps["scaler"].transform(X_imputed)
X_reduced = pipeline.named_steps["reduce"].transform(X_scaled)   # (79, 20) - all 20 RFE features

rfe_mask = pipeline.named_steps["reduce"].support_
selected_features = [name for name, keep in zip(all_column_names, rfe_mask) if keep]

print(f"RFE kept {len(selected_features)} / {len(all_column_names)} features.")
# (individual feature names are printed further below, ranked by SHAP
# importance, limited to the top TOP_N_TO_DISPLAY - see after SHAP is computed)

assert X_reduced.shape == (X.shape[0], RFE_N_FEATURES), (
    f"Expected X_reduced shape {(X.shape[0], RFE_N_FEATURES)}, got {X_reduced.shape}"
)

# SHAP - TreeExplainer on the fitted CatBoost model (exact, not approximated)
# Computed over ALL 20 RFE-selected features - the full feature set.

catboost_model = pipeline.named_steps["clf"]

explainer = shap.TreeExplainer(catboost_model)
shap_values = explainer.shap_values(X_reduced)   # shape (79, 20) - full, all 20 features

print(f"\nSHAP values computed: shape {shap_values.shape}")

# RANK ALL 20 FEATURES BY MEAN |SHAP|

mean_abs_shap_all = np.abs(shap_values).mean(axis=0)
ranked_idx = np.argsort(mean_abs_shap_all)[::-1]   # descending order, over all 20

print(f"\nTop {TOP_N_TO_DISPLAY} of {len(selected_features)} RFE-selected features "
      f"(ranked by mean |SHAP|):")
for rank, idx in enumerate(ranked_idx[:TOP_N_TO_DISPLAY], start=1):
    print(f"  {rank:2d}. {selected_features[idx]:35s} "
          f"(mean |SHAP| = {mean_abs_shap_all[idx]:.4f})")


top_idx = ranked_idx[:TOP_N_TO_DISPLAY]
shap_values_top10 = shap_values[:, top_idx]          # (79, 10)
X_reduced_top10 = X_reduced[:, top_idx]              # (79, 10)
top10_feature_names = [selected_features[i] for i in top_idx]

with open(os.path.join(OUTPUT_DIR, "shap_top10_features.txt"), "w") as f:
    f.write("\n".join(top10_feature_names))


np.save(os.path.join(OUTPUT_DIR, "shap_values_correlation_catboost.npy"), shap_values)

with open(os.path.join(OUTPUT_DIR, "shap_selected_features.txt"), "w") as f:
    f.write("\n".join(selected_features))

per_subject_df = pd.DataFrame(shap_values, columns=selected_features)
per_subject_df.insert(0, "label", y)
per_subject_df.insert(0, "subject_index", np.arange(len(y)))
per_subject_df.to_csv(
    os.path.join(OUTPUT_DIR, "shap_per_subject_values.csv"), index=False
)

print(f"\nSaved full 20-feature SHAP values, selected feature list, "
      f"top-10 feature list, and per-subject CSV to:\n  {OUTPUT_DIR}")


# bar plot

plt.figure(figsize=(9, 7))
shap.summary_plot(
    shap_values_top10, X_reduced_top10, feature_names=top10_feature_names,
    plot_type="bar", show=False
)
plt.title(f"Global Feature Importance (mean |SHAP|) - Top {TOP_N_TO_DISPLAY}\n"
          f"{MEASURE} + CatBoost, RFE({RFE_N_FEATURES})+SMOTE pipeline")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "shap_summary_bar.png"), dpi=200)
plt.close()

# beeswarm summary plot

plt.figure(figsize=(9, 7))
shap.summary_plot(
    shap_values_top10, X_reduced_top10, feature_names=top10_feature_names,
    show=False
)
plt.title(f"SHAP Summary (importance + direction) - Top {TOP_N_TO_DISPLAY}\n"
          f"{MEASURE} + CatBoost, RFE({RFE_N_FEATURES})+SMOTE pipeline")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "shap_summary_beeswarm.png"), dpi=200)
plt.close()

# DEPENDENCE PLOT - top feature's value vs its SHAP contribution


top_feature_idx_full = int(ranked_idx[0])   # index into the FULL 20-feature arrays
top_feature_name = selected_features[top_feature_idx_full]

plt.figure(figsize=(8, 6))
shap.dependence_plot(
    top_feature_idx_full, shap_values, X_reduced,
    feature_names=selected_features, show=False
)
plt.title(f"SHAP Dependence: {top_feature_name}\n(top-ranked feature by mean |SHAP|)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "shap_dependence_top_feature.png"), dpi=200)
plt.close()

print(f"\nTop feature by mean |SHAP|: {top_feature_name} "
      f"(mean |SHAP| = {mean_abs_shap_all[top_feature_idx_full]:.4f})")

print("\nSaved plots (bar and beeswarm show EXACTLY the top "
      f"{TOP_N_TO_DISPLAY} of {RFE_N_FEATURES} RFE-selected features):")
print(f"  - shap_summary_bar.png")
print(f"  - shap_summary_beeswarm.png")
print(f"  - shap_dependence_top_feature.png")

print(f"\nAll SHAP explainability outputs saved to: {OUTPUT_DIR}")
print("This folder is separate from data/features_paper_method/ and "
      "results_rfe_smote_final/ - no existing files were modified or overwritten.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded Correlation features: shape (79, 40)
Loaded labels: shape (79,), Control=42, ADHD=37

Fitting the Correlation + CatBoost (RFE+SMOTE) pipeline on all 79 subjects (one-time refit for explainability - see caveat in the script docstring)...
done.

RFE kept 20 / 40 features.

SHAP values computed: shape (79, 20)

Top 10 of 20 RFE-selected features (ranked by mean |SHAP|):
   1. cell6_Cz-F4_LowerAlpha              (mean |SHAP| = 0.6274)
   2. cell7_O1-F4_Delta                   (mean |SHAP| = 0.5566)
   3. cell8_Fz-F4_Delta                   (mean |SHAP| = 0.3650)
   4. cell6_Cz-F4_HigherAlpha             (mean |SHAP| = 0.3198)
   5. cell3_Cz-F4_Delta                   (mean |SHAP| = 0.2837)
   6. cell3_Cz-F4_HigherAlpha             (mean |SHAP| = 0.2757)
   7. cell2_Cz-F4_LowerAlpha              (mean |SHAP| = 0.2682)
   8. cell1_Cz-F4_Delta                

<Figure size 800x600 with 0 Axes>